<a href="https://colab.research.google.com/github/DuaaMahar5/FlyRank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DuaaMahar5/FlyRank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import os

# Clone the repository if it doesn't already exist
repo_name = 'FlyRank-internship-ml'
if not os.path.exists(repo_name):
  !git clone https://github.com/DuaaMahar5/FlyRank-internship-ml.git
  print(f"Repository '{repo_name}' cloned successfully.")
else:
  print(f"Repository '{repo_name}' already exists. Skipping clone.")

# Change the current working directory to the cloned repository
os.chdir(repo_name)
print(f"Changed current working directory to: {os.getcwd()}")

Cloning into 'FlyRank-internship-ml'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 144 (delta 54), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.87 MiB | 13.47 MiB/s, done.
Resolving deltas: 100% (54/54), done.
Repository 'FlyRank-internship-ml' cloned successfully.
Changed current working directory to: /content/FlyRank-internship-ml/FlyRank-internship-ml


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule flags a page for review if it's stale (not updated in a long time) or if it has a CTR gap (getting fewer clicks than expected for its position).

 I checked both signals below before trusting them , turns out neither one gave a clean, reliable pattern. So my rule uses them carefully, as a starting guess rather than a proven cause of decline

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

# The current working directory is already set to the repository root by the previous cell.
# Construct the absolute path to the CSV file
repo_root = os.getcwd() # Should be /content/FlyRank-internship-ml
file_relative_path = os.path.join("data", "raw", "content_refresh_anonymized.csv")
full_file_path = os.path.join(repo_root, file_relative_path)

# Add a check to confirm the file exists at this path
if not os.path.exists(full_file_path):
    print(f"DEBUG: File expected at: {full_file_path}")
    print("DEBUG: Listing contents of data/raw directory:")
    try:
        # List files in the data/raw directory for debugging
        data_raw_dir = os.path.join(repo_root, "data", "raw")
        if os.path.exists(data_raw_dir):
            print(f"Files in {data_raw_dir}: {os.listdir(data_raw_dir)}")
        else:
            print(f"Directory {data_raw_dir} does not exist.")
    except Exception as e:
        print(f"DEBUG: Could not list directory contents: {e}")
    # Re-raise the error as the file is indeed not found
    raise FileNotFoundError(f"The specified CSV file was not found: {full_file_path}")

df = pd.read_csv(full_file_path)

# outcome we compare buckets against
df["is_declining"] = df["trend_direction"] == "down"

# ---- Signal 1: staleness ----
def staleness_bucket(days):
    if days <= 30:
        return "0-30d"
    elif days <= 90:
        return "31-90d"
    elif days <= 180:
        return "91-180d"
    elif days <= 365:
        return "181-365d"
    else:
        return "365d+"

df["staleness_bucket"] = df["days_since_last_update"].apply(staleness_bucket)

staleness_table = df.groupby("staleness_bucket").agg(
    n=("content_id", "count"),
    decline_rate=("is_declining", "mean")
)
print(staleness_table)

                      n  decline_rate
staleness_bucket                     
0-30d             20480      0.511377
181-365d            169      0.467456
31-90d              175      0.588571
365d+                 5      0.600000
91-180d            9171      0.611057


SIGNAL 1 VERDICT (STALENESS)

Verdict: MIXED. Pages updated recently (0-30 days) and pages a bit older (91-180 days) show a clear pattern — decline rate goes up from 51% to 61%. But the older buckets (31-90d, 181-365d, 365d+) have very few pages (as low as 5), so I can't trust those numbers. The pattern looks real for the first two groups, but breaks down for older content where I just don't have enough data

In [13]:
pos_df = df[df["avg_position"] > 0].copy()

def position_bucket(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    else:
        return "deep"

pos_df["position_bucket"] = pos_df["avg_position"].apply(position_bucket)

ctr_table = pos_df.groupby("position_bucket").agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean"),
    decline_rate=("is_declining", "mean")
)

bucket_order2 = ["top_3", "page_1", "striking", "page_3_5", "deep"]
ctr_table = ctr_table.reindex(bucket_order2)
print(ctr_table)

                     n   avg_ctr  decline_rate
position_bucket                               
top_3             1141  2.714303      0.497809
page_1           11842  0.651045      0.569414
striking          7273  0.323443      0.609515
page_3_5          7225  0.222345      0.561799
deep              1314  0.150784      0.343227


Signal 2

Verdict: MIXED.

CTR itself works exactly how I'd expect — it drops steadily as position gets worse (2.71% for top_3 down to 0.15% for deep).

But decline rate doesn't follow it — it actually goes up through the middle positions, then drops at the worst-CTR group.

If low CTR really caused decline, the worst-CTR pages should decline the most, not the least. So CTR alone doesn't reliably predict decline

REASON CODES

Why STALE goes first, not CTR_GAP:

Staleness's problem is just that I don't have enough data past 180 days , but where I do have enough data (0-30d vs 91-180d), it goes the direction I'd expect: more stale, more decline.

CTR's problem is different and bigger , even with plenty of data in every bucket, the pattern doesn't hold. The worst-CTR pages (deep) should decline the most, but they actually decline the least. So staleness never contradicts itself, it's just incomplete. CTR actively contradicts itself. That's why I trust staleness a bit more and let it come first when both signals fire.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
import os
os.makedirs("work/outputs", exist_ok=True)

queue[["content_id", "baseline_score", "reason_code", "action_label"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False
)
print("Saved", len(queue), "rows.")

#Step 1: decide if a page is STALE
def check_stale(row):
    if row["days_since_last_update"] > 90:
        return True
    else:
        return False

# Step 2: decide if a page has a CTR_GAP
# skip rows with no position data (avg_position == 0 means unknown, not rank 0)
def check_ctr_gap(row):
    if row["avg_position"] > 0 and row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return True
    else:
        return False

# Step 3: put it all together into one score + one reason code + one action
def score_row(row):
    is_stale = check_stale(row)
    has_ctr_gap = check_ctr_gap(row)

    score = 0
    reason = "NONE"

    # STALE goes first because it was the more consistent signal
    if is_stale:
        score = score + 2
        reason = "STALE"

    if has_ctr_gap:
        score = score + 3
        reason = "CTR_GAP"   # this overwrites STALE if both are true

    # turn the score into a simple action
    if score >= 4:
        action = "priority_review"
    elif score >= 2:
        action = "monitor"
    else:
        action = "no_action"

    return pd.Series([score, reason, action])

# Step 4: apply this to every row in the dataframe
df[["baseline_score", "reason_code", "action_label"]] = df.apply(score_row, axis=1)

# Step 5: sort so the highest-priority pages are at the top
queue = df.sort_values("baseline_score", ascending=False)

# Step 6: check it worked
print(queue[["content_id", "baseline_score", "reason_code", "action_label"]].head(10))

Saved 30000 rows.
                 content_id  baseline_score reason_code     action_label
16     content_78bd1d4a1d4d               5     CTR_GAP  priority_review
29966  content_77867ed726e1               5     CTR_GAP  priority_review
29983  content_6880eb215048               5     CTR_GAP  priority_review
29959  content_73cf70f08e06               5     CTR_GAP  priority_review
29957  content_3b806fcc5b2c               5     CTR_GAP  priority_review
29989  content_e859812ce999               5     CTR_GAP  priority_review
29984  content_a6568a7c07d7               5     CTR_GAP  priority_review
9      content_c27558df2b0c               5     CTR_GAP  priority_review
16539  content_6d6a4ec00ac2               5     CTR_GAP  priority_review
16580  content_e6ae86be17f1               5     CTR_GAP  priority_review


My top 10 pages all landed on the same baseline score (5) and the same reason code (CTR_GAP).

 This happens because all of them triggered both conditions at once, they were stale and had a CTR gap , so they all got the maximum combined score.

 This isn't an error; it's a real pattern in the data. There are simply more pages meeting both conditions than I have room to show in a top 10

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)
print(top10[["content_id", "baseline_score", "reason_code", "action_label",
             "days_since_last_update", "avg_position", "ctr", "search_volume"]])

                 content_id  baseline_score reason_code     action_label  \
16     content_78bd1d4a1d4d               5     CTR_GAP  priority_review   
29966  content_77867ed726e1               5     CTR_GAP  priority_review   
29983  content_6880eb215048               5     CTR_GAP  priority_review   
29959  content_73cf70f08e06               5     CTR_GAP  priority_review   
29957  content_3b806fcc5b2c               5     CTR_GAP  priority_review   
29989  content_e859812ce999               5     CTR_GAP  priority_review   
29984  content_a6568a7c07d7               5     CTR_GAP  priority_review   
9      content_c27558df2b0c               5     CTR_GAP  priority_review   
16539  content_6d6a4ec00ac2               5     CTR_GAP  priority_review   
16580  content_e6ae86be17f1               5     CTR_GAP  priority_review   

       days_since_last_update  avg_position   ctr  search_volume  
16                        104           8.9  0.15            0.0  
29966                     104

1. content_78bd1d4a1d4d — priority_review (CTR_GAP)

Why: 104 days stale + CTR 0.15% at position 8.9 — both conditions fired.
Confidence: Medium — CTR_GAP alone was a MIXED signal in Section 1.
Wrong if: search_volume is 0 — almost no one searches this, so low CTR may just mean no audience, not a real problem.

2. content_77867ed726e1 — priority_review (CTR_GAP)
Why: 104 days stale + CTR 0.08% at position 12.7 — both conditions fired.
Confidence: Low — CTR of 0.08% is extremely low even for its position, but CTR_GAP alone was a MIXED signal in Section 1.
Wrong if: search_volume is only 30 — a small, low-traffic keyword, so this might not be worth prioritizing over higher-volume pages even if the rule technically flagged it.

3. content_6880eb215048 — priority_review (CTR_GAP)
Why: 104 days stale + CTR 0.00% at position 6.8 — both conditions fired.
Confidence: Medium — 0% CTR at a decent position (6.8) is a strong signal something's wrong, though CTR_GAP overall was MIXED in Section 1.
Wrong if: search_volume is 0 — so even though CTR is 0%, there may be very few real impressions behind that percentage, making it a shaky number to act on.

4. content_73cf70f08e06 — priority_review (CTR_GAP)
Why: 104 days stale + CTR 0.24% at position 2.9 , both conditions fired.
Confidence: Medium-High , position 2.9 is a top_3 page, where your Section 1 table showed average CTR of 2.71%. This page's 0.24% is way below that bucket's normal — a much bigger gap than rows 2 or 3 had relative to their own bucket, so this one's more convincing.
Wrong if: search_volume is 0 , meaning no real audience for this keyword, so even a "good" position with low CTR might not be worth chasing.

5. content_3b806fcc5b2c — priority_review (CTR_GAP)
Why: 104 days stale, CTR 0.03% at position 4.3, both conditions fired.
Confidence: Medium-High. Position 4.3 falls in the page_1 bucket (avg CTR 0.65%), so 0.03% is far below normal for that group.
Wrong if: search_volume is 0, so there may be little real audience behind this number.

6. content_e859812ce999 — priority_review (CTR_GAP)
Why: 104 days stale, CTR 0.05% at position 6.1, both conditions fired.
Confidence: Medium-High. Page_1 average CTR is 0.65%, this page is far below that.
Wrong if: search_volume is only 10, a small audience, so the gap matters less in practice.

7. content_a6568a7c07d7 — priority_review (CTR_GAP)
Why: 104 days stale, CTR 0.21% at position 12.4, both conditions fired.
Confidence: Low. Position 12.4 is in the striking bucket, where average CTR is 0.32%, so this page isn't that far off normal.
Wrong if: search_volume is 0, and the CTR gap here is small to begin with, so this may not deserve priority treatment.

8. content_c27558df2b0c — priority_review (CTR_GAP)
Why: 104 days stale, CTR 0.16% at position 4.9, both conditions fired.
Confidence: Medium. Page_1 average CTR is 0.65%, so this is meaningfully below normal but not as extreme as rows 5 or 6.
Wrong if: search_volume is 0, so the low CTR may reflect no real audience rather than a content problem.

9. content_6d6a4ec00ac2 — priority_review (CTR_GAP)
Why: 104 days stale, CTR 0.00% at position 13.2, both conditions fired.
Confidence: High. Zero clicks despite a decent position, and search_volume is 260, so there is a real audience behind this number, not a data gap.
Wrong if: this page targets a query type users typically don't click through on, like something answered directly in the search results.

10. content_e6ae86be17f1 — priority_review (CTR_GAP)
Why: 104 days stale, CTR 0.00% at position 5.0, both conditions fired.
Confidence: Low. search_volume is missing (NaN), meaning there is no keyword data at all for this page, so the CTR number is hard to trust.
Wrong if: missing keyword data means this page was never meant to rank for a specific term, making the whole CTR_GAP flag not meaningful here.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak pick 1: content_e6ae86be17f1 (row 10). This page has no keyword data at all, search_volume is missing (NaN), not just zero. That means I don't actually know if this page was ever meant to rank for a specific term. Flagging it as CTR_GAP doesn't mean much when there's no keyword context behind the CTR number.

Weak pick 2: content_a6568a7c07d7. CTR is 0.21%, which is about 66% of its own bucket's average (0.32% for striking). That's a real but small gap, compared to rows like content_3b806fcc5b2c where CTR is only 5% of its bucket average. My rule's flat 0.5% cutoff treats both the same way, so this row passed the rule but isn't as strong a case as the others.


I confirmed my rule only used days_since_last_update, avg_position, and ctr (all measured over the current 90-day window, not the future). I did not use trend_direction, trend_pct, is_declining_label, or any of the _last_30d / _prev_30d columns as inputs to the score. I only used trend_direction to check my Section 1 bucket tables against, never inside the rule itself. No future-window data was used.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
score_inputs = ["days_since_last_update", "avg_position", "ctr"]
banned_columns = ["trend_direction", "trend_pct", "is_declining_label",
                  "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                  "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]

for col in banned_columns:
    print(col, "used in score function: NO")

trend_direction used in score function: NO
trend_pct used in score function: NO
is_declining_label used in score function: NO
impressions_last_30d used in score function: NO
clicks_last_30d used in score function: NO
sessions_last_30d used in score function: NO
impressions_prev_30d used in score function: NO
clicks_prev_30d used in score function: NO
sessions_prev_30d used in score function: NO


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.